In [39]:
import numpy as np
from typing import List, Tuple

class NeuralNetwork:
    """
    Implements a feedforward neural network for binary classification.

    Args:
        layer_sizes: List of integers for each layer's neuron count.
        learning_rate: Step size for gradient descent (default 0.01).
    """
    def __init__(self, layer_sizes: List[int], learning_rate: float = 0.01):
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.num_layers = len(layer_sizes)
        self.weights = []
        self.biases = []
        for i in range(self.num_layers - 1):
            w = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2.0 / layer_sizes[i])
            b = np.zeros((1, layer_sizes[i+1]))
            self.weights.append(w)
            self.biases.append(b)

    def relu(self, x: np.ndarray) -> np.ndarray:
        """Apply ReLU elementwise."""
        return np.maximum(0, x)

    def relu_derivative(self, x: np.ndarray) -> np.ndarray:
        """Returns derivative of ReLU for input x."""
        return (x > 0).astype(float)

    def sigmoid(self, x: np.ndarray) -> np.ndarray:
        """Sigmoid activation: 1 / (1 + exp(-x))"""
        x_clipped = np.clip(x, -500, 500)
        return (1 / (1 + np.exp(-x_clipped)))

    def sigmoid_derivative(self, x: np.ndarray) -> np.ndarray:
        """Returns derivative of sigmoid for input x."""
        sigmoid = self.sigmoid(x)
        return sigmoid * (1 - sigmoid)

    def forward(self, X: np.ndarray) -> Tuple[np.ndarray, List[np.ndarray], List[np.ndarray]]:
        """Forward pass: returns (output, [activations], [pre_activations])."""
        # TODO: Implement forward propagation
        # 1. Initialize activations list with input X
        activations = [X]
        pre_activations = []

        # 2. For each layer: z = activations[-1] @ weights[i] + biases[i]
        for i in range(self.num_layers - 1):
            z = activations[-1] @ self.weights[i] + self.biases[i]
            # 3. Apply ReLU for hidden layers, sigmoid for output layer
            if i < self.num_layers - 2:
                a = self.relu(z)
            else:
                a = self.sigmoid(z)
            # 4. Store both pre-activations (z) and activations (after activation function)
            pre_activations.append(z)
            activations.append(a)

        # 5. Return final output, all activations, all pre-activations
        return activations[-1], activations, pre_activations

    def backward(
        self,
        X: np.ndarray,
        y: np.ndarray,
        activations: List[np.ndarray],
        pre_activations: List[np.ndarray]
    ) -> Tuple[List[np.ndarray], List[np.ndarray]]:
        """Backward pass: compute and return gradients."""
        m = X.shape[0]  # Batch size

        # Ensure y has correct shape (m, 1)
        if y.ndim == 1:
            y = y.reshape(-1, 1)

        # Initialize gradient lists with the same order as weights/biases
        weight_gradients = [None] * (self.num_layers - 1)
        bias_gradients = [None] * (self.num_layers - 1)

        # Output layer error (Sigmoid + Binary Cross Entropy)
        dz = activations[-1] - y

        # Iterate from the last layer to the first hidden layer
        for i in range(self.num_layers - 2, -1, -1):
            # Weight gradient
            weight_gradients[i] = activations[i].T @ dz / m

            # Bias gradient
            bias_gradients[i] = np.sum(dz, axis=0, keepdims=True) / m

            # Propagate error to previous layer (if not input layer)
            if i > 0:
                da = dz @ self.weights[i].T
                dz = da * self.relu_derivative(pre_activations[i - 1])

        # Return lists of weight_gradients and bias_gradients
        return weight_gradients, bias_gradients

    def update_weights(
        self,
        weight_gradients: List[np.ndarray],
        bias_gradients: List[np.ndarray]
    ) -> None:
        """Update model weights using gradients with gradient clipping."""
        # Gradient clipping to prevent exploding gradients
        max_grad_norm = 5.0

        for i in range(self.num_layers - 1):
            # Clip gradients
            weight_gradients[i] = np.clip(weight_gradients[i], -max_grad_norm, max_grad_norm)
            bias_gradients[i] = np.clip(bias_gradients[i], -max_grad_norm, max_grad_norm)

            # Update weights and biases
            self.weights[i] -= self.learning_rate * weight_gradients[i]
            self.biases[i] -= self.learning_rate * bias_gradients[i]

    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict binary classes for input.
        Returns: np.ndarray of shape (N,), values 0 or 1.
        """
        # 1. Run forward pass to get output probabilities
        output, _, _ = self.forward(X)
        # 2. Convert probabilities to binary predictions: (output > 0.5).astype(int)
        predictions = (output > 0.5).astype(int)
        # 3. Return flattened array of shape (N,)
        return predictions.flatten()

    def compute_loss(self, y_pred: np.ndarray, y_true: np.ndarray) -> float:
        """Compute binary cross-entropy loss."""
        # Ensure both arrays have same shape
        if y_true.ndim == 1:
            y_true = y_true.reshape(-1, 1)
        if y_pred.ndim == 1:
            y_pred = y_pred.reshape(-1, 1)

        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

In [40]:
import copy
import numpy as np
from typing import Tuple

class Trainer:
    def __init__(self, model: NeuralNetwork):
        self.model = model
        self.training_losses = []
        self.validation_losses = []
        self.training_accuracies = []
        self.validation_accuracies = []

    def train_epoch(self, X_train: np.ndarray, y_train: np.ndarray) -> Tuple[float, float]:
        """Train the model for one epoch."""
        # 1. Forward pass: output, activations, pre_activations = model.forward(X_train)
        output, activations, pre_activations = self.model.forward(X_train)
        # 2. Compute loss: loss = model.compute_loss(output, y_train)
        loss = self.model.compute_loss(output, y_train)
        # 3. Backward pass: weight_grads, bias_grads = model.backward(...)
        weight_grads, bias_grads = self.model.backward(X_train, y_train, activations, pre_activations)
        # 4. Update weights: model.update_weights(weight_grads, bias_grads)
        self.model.update_weights(weight_grads, bias_grads)
        # 5. Calculate accuracy and return (loss, accuracy)
        predictions = self.model.predict(X_train)
        accuracy = np.mean(predictions == y_train.flatten())
        return loss, accuracy

    def validate(self, X_val: np.ndarray, y_val: np.ndarray) -> Tuple[float, float]:
        """Evaluate model on validation data."""
        # 1. Forward pass only (no weight updates)
        output, _, _ = self.model.forward(X_val)
        # 2. Compute loss and predictions
        loss = self.model.compute_loss(output, y_val)
        predictions = (output > 0.5).astype(int).flatten()
        # 3. Calculate accuracy and return (loss, accuracy)
        accuracy = np.mean(predictions == y_val.flatten())
        return loss, accuracy

    def train(self, X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, epochs: int = 100, early_stopping_patience: int = 10) -> dict:
        """Train with early stopping, returns history dict."""
        # TODO: Implement main training loop with early stopping
        # 1. Initialize best_val_loss = inf, patience_counter = 0
        best_val_loss = float('inf')
        patience_counter = 0

        self.training_losses = []
        self.validation_losses = []
        self.training_accuracies = []
        self.validation_accuracies = []

        # 2. For each epoch: train_epoch(), validate(), store metrics
        for epoch in range(epochs):
            train_loss, train_accuracy = self.train_epoch(X_train, y_train)
            val_loss, val_accuracy = self.validate(X_val, y_val)
            self.training_losses.append(train_loss)
            self.validation_losses.append(val_loss)
            self.training_accuracies.append(train_accuracy)
            self.validation_accuracies.append(val_accuracy)
            # 3. Early stopping: if val_loss improves, reset patience; else increment
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_weights = copy.deepcopy(self.model.weights)
                best_biases = copy.deepcopy(self.model.biases)
            else:
                patience_counter += 1
            # 4. Stop if patience_counter >= early_stopping_patience
            if patience_counter >= early_stopping_patience:
                print(f"Early stopping triggered after {epoch + 1} epochs.")
                break
        # 5. Return history dict with losses and accuracies
        return {
            "training_losses": self.training_losses,
            "validation_losses": self.validation_losses,
            "training_accuracies": self.training_accuracies,
            "validation_accuracies": self.validation_accuracies,
            "best_val_loss": best_val_loss,
            "epochs_trained": epoch + 1,
            "weights": best_weights,
            "biases": best_biases
        }

def normalize_features(X: np.ndarray) -> np.ndarray:
    """Normalize features to zero mean/unit variance."""
    # Formula: (X - mean) / std
    epsilon = 1e-8
    X_mean = np.mean(X, axis=0, keepdims=True)
    X_std = np.std(X, axis=0, keepdims=True) + epsilon  # Add small epsilon to avoid division by zero
    X_normalized = (X - X_mean) / X_std
    return X_normalized

def split_data(X: np.ndarray, y: np.ndarray, train_ratio: float = 0.7, val_ratio: float = 0.15) -> Tuple:
    """Split data into train, validation, and test sets with fixed seed."""
    # 1. Get total samples: n_samples = X.shape[0]
    n_samples = X.shape[0]
    # 2. Use fixed seed for reproducible splits: rng = np.random.RandomState(42)
    rng = np.random.RandomState(42)
    # 3. Create random permutation: indices = rng.permutation(n_samples)
    indices = rng.permutation(n_samples)
    # 4. Calculate split sizes: train_size = int(train_ratio * n_samples), etc.
    train_size = int(train_ratio * n_samples)
    val_size = int(val_ratio * n_samples)
    # 5. Create train/val/test indices and return split data
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]
    # 6. Return (X_train, y_train, X_val, y_val, X_test, y_test)
    return (X[train_indices], y[train_indices],
            X[val_indices], y[val_indices],
            X[test_indices], y[test_indices])

def load_data() -> Tuple[np.ndarray, np.ndarray]:
    """Load the customer churn dataset from CSV file."""
    # 1. Import pathlib.Path and pandas (or numpy for fallback)
    from pathlib import Path
    import pandas as pd
    # 2. Load dataset from data/customer_churn.csv
    data_path = Path("data.csv")
    # 3. Separate features and labels:
    #    Features: account_age, purchase_frequency, avg_order_value, support_tickets, days_since_last_purchase
    #    Label: churn
    df = pd.read_csv(data_path)
    X = (
        df[
            [
                "account_age",
                "purchase_frequency",
                "avg_order_value",
                "support_tickets",
                "days_since_last_purchase",
            ]
        ]
        .astype(np.float64)
        .values
    )
    y = df["churn"].values.reshape(-1,1)
    # 4. Return (X, y) where X is feature matrix and y is label vector
    return X, y


In [41]:
"""utils.py - Helper metrics and plotting utilities for neural network challenge."""
import numpy as np
from typing import Tuple

def accuracy(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """Compute classification accuracy."""
    return np.mean(y_pred == y_true)

def precision_recall_f1(y_pred: np.ndarray, y_true: np.ndarray) -> Tuple[float, float, float]:
    """Compute (precision, recall, f1-score)."""
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    # 1. Precision = TP / (TP + FP), Recall = TP / (TP + FN)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    # 2. Recall = TP / (TP + FN)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    # 3. F1 = 2 * precision * recall / (precision + recall)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

def confusion_matrix(y_pred: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    """Compute confusion matrix: TN/FP/FN/TP."""
    tn = np.sum((y_pred == 0) & (y_true == 0))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    tp = np.sum((y_pred == 1) & (y_true == 1))
    return np.array([[tn, fp], [fn, tp]])

def plot_decision_boundary(model, X: np.ndarray, y: np.ndarray, feature_names: list, title: str = "Decision Boundary") -> None:
    """Optional: Plot 2D decision boundary for visualization."""
    import matplotlib.pyplot as plt

    # Create a mesh grid for the feature space
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.01),
                         np.arange(y_min, y_max, 0.01))

    # Predict on the mesh grid
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot the decision boundary
    plt.contourf(xx, yy, Z, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o')
    plt.xlabel(feature_names[0])
    plt.ylabel(feature_names[1])
    plt.title(title)
    plt.show()

In [42]:
import pandas as pd

X, y = load_data()

model = NeuralNetwork(layer_sizes=[X.shape[1], 64, 32, 1], learning_rate=0.001)
trainer = Trainer(model)

X_train, y_train, X_val, y_val, X_test, y_test = split_data(X, y)

# CRITICAL FIX: Normalize using training statistics only
# Calculate mean and std from training data
train_mean = np.mean(X_train, axis=0, keepdims=True)
train_std = np.std(X_train, axis=0, keepdims=True) + 1e-8

# Apply same normalization to all sets
X_train = (X_train - train_mean) / train_std
X_val = (X_val - train_mean) / train_std
X_test = (X_test - train_mean) / train_std

history = trainer.train(X_train, y_train, X_val, y_val, epochs=100)

print(f"\nTraining completed after {history['epochs_trained']} epochs")
print(f"Best validation loss: {history['best_val_loss']:.4f}")


Training completed after 100 epochs
Best validation loss: 0.6689


/var/folders/jp/1hn6m8t16tq4_953r183nr0h0000gn/T/ipykernel_36779/2119031644.py:51: RuntimeWarning: divide by zero encountered in matmul
  z = activations[-1] @ self.weights[i] + self.biases[i]
/var/folders/jp/1hn6m8t16tq4_953r183nr0h0000gn/T/ipykernel_36779/2119031644.py:51: RuntimeWarning: overflow encountered in matmul
  z = activations[-1] @ self.weights[i] + self.biases[i]
/var/folders/jp/1hn6m8t16tq4_953r183nr0h0000gn/T/ipykernel_36779/2119031644.py:51: RuntimeWarning: invalid value encountered in matmul
  z = activations[-1] @ self.weights[i] + self.biases[i]
/var/folders/jp/1hn6m8t16tq4_953r183nr0h0000gn/T/ipykernel_36779/2119031644.py:88: RuntimeWarning: divide by zero encountered in matmul
  weight_gradients[i] = activations[i].T @ dz / m
/var/folders/jp/1hn6m8t16tq4_953r183nr0h0000gn/T/ipykernel_36779/2119031644.py:88: RuntimeWarning: overflow encountered in matmul
  weight_gradients[i] = activations[i].T @ dz / m
/var/folders/jp/1hn6m8t16tq4_953r183nr0h0000gn/T/ipykernel_367